In [1]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import XGBoost dan Sklearn Metrics
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Setup tema plot
sns.set_theme(style="whitegrid", palette="muted")
import warnings
warnings.filterwarnings('ignore')

print("Library untuk pemodelan XGBoost berhasil dimuat!")

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# ==========================================
# 2. LOAD PRE-PROCESSED DATA
# ==========================================
# Ganti nama file jika letaknya di folder berbeda
train_df = pd.read_csv('train_preprocessed_smote.csv')
test_df = pd.read_csv('test_preprocessed.csv')

# Memisahkan Fitur (X) dan Target (y)
X_train = train_df.drop(columns=['addicted_label'])
y_train = train_df['addicted_label']

X_test = test_df.drop(columns=['addicted_label'])
y_test = test_df['addicted_label']

print(f"Dimensi Data Latih (X_train): {X_train.shape}")
print(f"Dimensi Data Uji (X_test): {X_test.shape}")
print("Data siap dilatih!")

In [ ]:
# ==========================================
# 3. TRAINING XGBOOST MODEL
# ==========================================
# Inisialisasi Model XGBoost
xgb_model = xgb.XGBClassifier(
    learning_rate=0.1,        # Kecepatan belajar model
    n_estimators=100,         # Jumlah pohon (trees)
    max_depth=5,              # Kedalaman maksimal setiap pohon
    subsample=0.8,            # Mengambil 80% sampel untuk setiap pohon (mencegah overfitting)
    colsample_bytree=0.8,     # Mengambil 80% fitur untuk setiap pohon
    random_state=42,          # Agar hasil bisa direplikasi
    eval_metric='logloss'     # Metrik evaluasi internal XGBoost
)

# Melatih Model
print("Sedang melatih model XGBoost... Mohon tunggu.")
xgb_model.fit(X_train, y_train)
print("Pelatihan model XGBoost selesai!")

In [ ]:
# ==========================================
# 4. EVALUASI KINERJA MODEL
# ==========================================
# Memprediksi data uji
y_pred_xgb = xgb_model.predict(X_test)

# 1. Classification Report
print("=== CLASSIFICATION REPORT (XGBOOST) ===")
print(classification_report(y_test, y_pred_xgb, target_names=['Tidak Kecanduan (0)', 'Kecanduan (1)']))

# 2. Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pred: Tidak', 'Pred: Kecanduan'], 
            yticklabels=['Aktual: Tidak', 'Aktual: Kecanduan'])
plt.title('Confusion Matrix - XGBoost', fontsize=14, fontweight='bold')
plt.xlabel('Nilai Prediksi')
plt.ylabel('Nilai Aktual')
plt.tight_layout()
plt.show()

# Insight untuk jurnal
akurasi = accuracy_score(y_test, y_pred_xgb)
print(f"\nAkurasi Keseluruhan XGBoost: {akurasi * 100:.2f}%")

In [ ]:
# ==========================================
# 5. FEATURE IMPORTANCE VISUALIZATION
# ==========================================
# Mengambil nilai kepentingan fitur dari model
importances = xgb_model.feature_importances_
feature_names = X_train.columns

# Membuat DataFrame untuk di-sorting
feature_imp_df = pd.DataFrame({
    'Fitur': feature_names,
    'Tingkat_Kepentingan': importances
}).sort_values(by='Tingkat_Kepentingan', ascending=False)

# Visualisasi Barplot
plt.figure(figsize=(10, 8))
sns.barplot(x='Tingkat_Kepentingan', y='Fitur', data=feature_imp_df, palette='viridis')
plt.title('Feature Importance (XGBoost)', fontsize=15, fontweight='bold')
plt.xlabel('Skor Kepentingan (F-Score)')
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.show()

print("=== 5 FITUR PALING BERPENGARUH ===")
print(feature_imp_df.head(5).to_string(index=False))